In [2]:
# ============================================================
# STAGE 1 — CORRECT SAE STAY-LEVEL PREDICTION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

ROOT = Path(
    "/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning"
)

DATA = ROOT / "sae_mimiciv_v2" / "data"
RESULTS = ROOT / "sae_mimiciv_v2" / "results"

RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

INPUT = DATA / "sepsis_sae_multimodal_hourly.csv"

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_csv(INPUT)

print("=" * 60)
print("CORRECTED STAGE 1 — SAE STAY-LEVEL PREDICTION")
print("=" * 60)

print("Dataset:", df.shape)
print("Patients:", df["subject_id"].nunique())
print("ICU stays:", df["stay_id"].nunique())

# ------------------------------------------------------------
# 2. STAY-LEVEL SAE LABEL
# ------------------------------------------------------------

stay_labels = (
    df.groupby("stay_id")["sae"]
    .max()
    .rename("sae_target")
    .reset_index()
)

df = df.drop(
    columns=["sae"],
    errors="ignore"
).merge(
    stay_labels,
    on="stay_id",
    how="left"
)

print("\nStay-level target:")
print(
    df[
        ["stay_id", "sae_target"]
    ]
    .drop_duplicates()["sae_target"]
    .value_counts()
)

print(
    "\nSAE stays:",
    df.loc[
        df["sae_target"] == 1,
        "stay_id"
    ].nunique()
)

print(
    "Non-SAE stays:",
    df.loc[
        df["sae_target"] == 0,
        "stay_id"
    ].nunique()
)

# ------------------------------------------------------------
# 3. PATIENT-LEVEL SPLIT
# ------------------------------------------------------------

patients = (
    df["subject_id"]
    .drop_duplicates()
    .tolist()
)

rng = np.random.RandomState(42)
rng.shuffle(patients)

n = len(patients)

n_train = int(n * 0.60)
n_val = int(n * 0.20)

train_patients = set(
    patients[:n_train]
)

val_patients = set(
    patients[
        n_train:n_train + n_val
    ]
)

test_patients = set(
    patients[
        n_train + n_val:
    ]
)

train = df[
    df["subject_id"].isin(train_patients)
].copy()

validation = df[
    df["subject_id"].isin(val_patients)
].copy()

test = df[
    df["subject_id"].isin(test_patients)
].copy()

# ------------------------------------------------------------
# 4. LEAKAGE CHECK
# ------------------------------------------------------------

assert len(
    train_patients & val_patients
) == 0

assert len(
    train_patients & test_patients
) == 0

assert len(
    val_patients & test_patients
) == 0

print(
    "\nPATIENT LEAKAGE CHECK: PASSED"
)

# ------------------------------------------------------------
# 5. IMPORTANT:
# COLLAPSE HOURLY DATA TO ONE ROW PER ICU STAY
# ------------------------------------------------------------
#
# We cannot treat thousands of hourly rows from one stay as
# independent training examples when the label is stay-level.
#
# Use the first available ICU observation for prediction.
# This avoids leaking future information from the same stay.

exclude = {
    "subject_id",
    "hadm_id",
    "stay_id",
    "hour_time",
    "hour",
    "sae_target"
}

candidate_features = [
    c for c in df.columns
    if c not in exclude
    and pd.api.types.is_numeric_dtype(
        df[c]
    )
]

# First hour of each ICU stay
train_stay = (
    train
    .sort_values(
        ["stay_id", "hour"]
    )
    .groupby("stay_id", as_index=False)
    .first()
)

val_stay = (
    validation
    .sort_values(
        ["stay_id", "hour"]
    )
    .groupby("stay_id", as_index=False)
    .first()
)

test_stay = (
    test
    .sort_values(
        ["stay_id", "hour"]
    )
    .groupby("stay_id", as_index=False)
    .first()
)

print("\nSTAY-LEVEL DATA")
print(
    "TRAIN:",
    train_stay.shape
)

print(
    "VALIDATION:",
    val_stay.shape
)

print(
    "TEST:",
    test_stay.shape
)

# ------------------------------------------------------------
# 6. FEATURES
# ------------------------------------------------------------

features = [
    c for c in candidate_features
    if train_stay[c].notna().any()
]

X_train = train_stay[features].copy()
X_val = val_stay[features].copy()
X_test = test_stay[features].copy()

y_train = train_stay[
    "sae_target"
].astype(int)

y_val = val_stay[
    "sae_target"
].astype(int)

y_test = test_stay[
    "sae_target"
].astype(int)

# ------------------------------------------------------------
# 7. TRAIN-ONLY MEDIAN IMPUTATION
# ------------------------------------------------------------

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_val = X_val.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

medians = X_train.median()

X_train = X_train.fillna(
    medians
).fillna(0)

X_val = X_val.fillna(
    medians
).fillna(0)

X_test = X_test.fillna(
    medians
).fillna(0)

# ------------------------------------------------------------
# 8. CLASS CHECK
# ------------------------------------------------------------

print("\nCLASS DISTRIBUTION")

print(
    "TRAIN:",
    y_train.value_counts().to_dict()
)

print(
    "VALIDATION:",
    y_val.value_counts().to_dict()
)

print(
    "TEST:",
    y_test.value_counts().to_dict()
)

# ------------------------------------------------------------
# 9. SAFETY CHECK
# ------------------------------------------------------------

if y_train.nunique() < 2:

    raise ValueError(
        "Training split contains only one class. "
        "With only 17 patients / 9 SAE stays, "
        "a random patient split can easily produce "
        "an invalid evaluation split. "
        "Use a stratified stay-level split."
    )

# ------------------------------------------------------------
# 10. CLASS-BALANCED RANDOM FOREST
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

# ------------------------------------------------------------
# 11. PROBABILITY PREDICTIONS
# ------------------------------------------------------------

train_risk = model.predict_proba(
    X_train
)[:, 1]

val_risk = model.predict_proba(
    X_val
)[:, 1]

test_risk = model.predict_proba(
    X_test
)[:, 1]

# ------------------------------------------------------------
# 12. METRICS
# ------------------------------------------------------------

def metrics(name, y, risk):

    if len(np.unique(y)) < 2:

        return {
            "split": name,
            "AUROC": np.nan,
            "AUPRC": np.nan,
            "Brier": np.nan
        }

    return {
        "split": name,
        "AUROC": roc_auc_score(
            y,
            risk
        ),
        "AUPRC": average_precision_score(
            y,
            risk
        ),
        "Brier": brier_score_loss(
            y,
            risk
        )
    }

results = pd.DataFrame([
    metrics(
        "TRAIN",
        y_train,
        train_risk
    ),
    metrics(
        "VALIDATION",
        y_val,
        val_risk
    ),
    metrics(
        "TEST",
        y_test,
        test_risk
    )
])

print("\n" + "=" * 60)
print("STAGE 1 RESULTS")
print("=" * 60)

display(results)

# ------------------------------------------------------------
# 13. SAVE RESULTS
# ------------------------------------------------------------

results.to_csv(
    RESULTS / "stage1_stay_level_metrics.csv",
    index=False
)

pd.DataFrame({
    "stay_id": test_stay["stay_id"],
    "subject_id": test_stay["subject_id"],
    "sae_target": y_test,
    "risk": test_risk
}).to_csv(
    RESULTS / "stage1_stay_level_test_risk.csv",
    index=False
)

# ------------------------------------------------------------
# 14. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTOP FEATURES")

display(
    importance.head(20)
)

importance.to_csv(
    RESULTS / "stage1_stay_level_feature_importance.csv",
    index=False
)

print("\n" + "=" * 60)
print("STAGE 1 COMPLETE")
print("=" * 60)
print(
    "Results saved to:",
    RESULTS
)

CORRECTED STAGE 1 — SAE STAY-LEVEL PREDICTION
Dataset: (3769, 63)
Patients: 17
ICU stays: 26

Stay-level target:
sae_target
0    17
1     9
Name: count, dtype: int64

SAE stays: 9
Non-SAE stays: 17

PATIENT LEAKAGE CHECK: PASSED

STAY-LEVEL DATA
TRAIN: (14, 63)
VALIDATION: (5, 63)
TEST: (7, 63)

CLASS DISTRIBUTION
TRAIN: {1: 7, 0: 7}
VALIDATION: {0: 3, 1: 2}
TEST: {0: 7}

STAGE 1 RESULTS


,split,AUROC,AUPRC,Brier
0,TRAIN,1.000000,1.000000,0.122836
1,VALIDATION,0.166667,0.366667,0.399793
2,TEST,NaN,NaN,NaN



TOP FEATURES


,feature,importance
33,resp_rate_delta,0.076422
51,temperature_mean3h,0.066382
3,heart_rate,0.057970
38,gcs_verbal_delta,0.053170
6,temperature,0.053053
48,heart_rate_mean3h,0.046661
50,spo2_mean3h,0.042917
32,heart_rate_delta,0.040278
25,creatinine_missing,0.038785
29,platelets_missing,0.038632



STAGE 1 COMPLETE
Results saved to: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_mimiciv_v2/results
